# 6.3. Parameter Initialization
D2L의 Parameter Initialization장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Parameter Initialization이란?

신경망에는 학습해야 하는 parameter W, b가 있다. 예를 들어서 

```text
입력 X -> Linear -> WX + b
```

그런데 학습 시작하기 전에는 W, b가 없다. 처음 시작할 값을 정해줘야 한다. 이것을 Parameter Initialization이라고 한다.

D2L에서도 프레임워크가 기본 초기화를 제공하지만, 필요에 따라 원하는 초기화 방법을 직접 지정할 수 있다고 설명한다.

## 2. 실습 모델 만들기

실습을 위해 간단한 신경망을 만들 것이다. 구조는 이렇다.

```text
입력 feature 4개
    |
Linear(4 -> 8)
    |
ReLU
    |
Linear(8 -> 1)
    |
출력 1개
```

In [2]:
import torch
from torch import nn

net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.rand(2, 4)

y_hat = net(X)

print(y_hat.shape)

torch.Size([2, 1])


## 3. PyTorch의 기본 초기화

`nn.Linear` 만들었다고 weight가 비어있는건 아니다.

`nn.Linear`를 생성하는 순간 PyTorch가 weight와 bias에 초기값을 넣어준다. 그래서 우리가 초기화하지 않아도 모델을 바로 사용할 수 있다.

In [ ]:
print(net[0].weight) # 초기화 된 값 확인
print(net[0].bias)

Parameter containing:
tensor([[ 0.3823,  0.4150, -0.1171,  0.4593],
        [-0.1096,  0.1009, -0.2434,  0.2936],
        [ 0.4408, -0.3668,  0.4346,  0.0936],
        [ 0.3694,  0.0677,  0.2411, -0.0706],
        [ 0.3854,  0.0739, -0.2334,  0.1274],
        [-0.2304, -0.0586, -0.2031,  0.3317],
        [-0.3947, -0.2305, -0.1412, -0.3006],
        [ 0.0472, -0.4938,  0.4516, -0.4247]], requires_grad=True)
Parameter containing:
tensor([ 0.3860,  0.0832, -0.1624,  0.3090,  0.0779,  0.4040,  0.0547, -0.1577],
       requires_grad=True)


`torch.nn.init`에 별도 초기화 함수들도 준비되어 있다고 한다.

## 4. 정규분포 초기화

원하는 방식으로 parameter를 초기화할 수도 있다. 이번엔 Linear 층의 weight를 평균 = 0이고 표준편차 = 0.01 인 정규분포에서 뽑도록 한다. bias는 모두 0으로 만든다.

In [4]:
def init_normal(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(
            module.weight,
            mean=0,
            std=0.01
        )
        
        nn.init.zeros_(module.bias)

In [ ]:
net.apply(init_normal) # 모델에 적용

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [7]:
print(net[0].weight[0])
print(net[0].bias[0])

tensor([-0.0092,  0.0181,  0.0016,  0.0037], grad_fn=<SelectBackward0>)
tensor(0., grad_fn=<SelectBackward0>)


## 5. net.apply() 이해

`apply()`는 모델 내부의 모듈들을 하나씩 방문하면서 지정한 함수를 실행한다. 현재 모델은 이렇게 생겼다.

```text
Sequential
├── Linear(4, 8)
├── ReLU()
└── Linear(8, 1)
```

`init_normal()`이 각 module에 대해 실행된다. 함수 내부에는 

    if isinstance(module, nn.Linear):

조건이 있어 Linear층만 초기화 한다고 한다. ReLU는 Parameter가 없어 아무 작업도 하지 않는다.

```text
Linear(4, 8) -> init_normal 실행
ReLU -> Linear가 아니라 무시
Linear(8, 1) -> init_normal 실행
```

## 6. 상수 초기화

parameter를 특정 값으로 설정하는 것도 가능하다.

예를 들어서 모든 weight를 1로 만들고 bias를 0으로 할 수도 있다.

In [8]:
def init_constant(module):
    if isinstance(module, nn.Linear):
        nn.init.constant_(module.weight, 1)
        nn.init.zeros_(module.bias)

net.apply(init_constant)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [9]:
print(net[0].weight[0])
print(net[0].bias[0])

tensor([1., 1., 1., 1.], grad_fn=<SelectBackward0>)
tensor(0., grad_fn=<SelectBackward0>)


실제 신경망에서 모든 weight를 같은 값으로 초기화하는 것은 일반적으로 좋은 초기화 방법이 아니다.

## 7. 층마다 다른 초기화 사용하기

모델 전체를 같은 방법으로 초기화할 필요는 없다.

예를 들어서

```text
첫 번째 Linear → Xavier 초기화
두 번째 Linear → 모든 weight를 42
```

층마다 다른 방법을 사용할 수 있다. `nn.Sequential`에서는 인덱스로 각 층에 접근할 수 있다.

In [10]:
def init_xavier(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)

def init_42(module):
    if isinstance(module, nn.Linear):
        nn.init.constant_(module.weight, 42)

In [11]:
net[0].apply(init_xavier)
net[2].apply(init_42)

Linear(in_features=8, out_features=1, bias=True)

In [12]:
print(net[0].weight[0])
print(net[2].weight)

tensor([-0.4830, -0.5999, -0.3894, -0.6189], grad_fn=<SelectBackward0>)
Parameter containing:
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]], requires_grad=True)


## 8. Xavier Initialization

Xavier 초기화는 신경망의 weight 크기가 너무 커지거나 너무 작아지지 않도록 초기값의 범위를 조절하는 방법이다.

층의 입력/출력 크기를 고려해 적절한 범위의 랜덤한 weight에서 시작하도록 하는 것이다. PyTorch에서는 다음처럼 사용할 수 있다.

    nn.init.xavier_uniform_(weight)

이전에 배운 numerical stability와 연결하면 된다.

좋지 않은 초기값 -> 값이나 gradient가 너무 커지거나 작아질 수 있다.
적절한 초기값 -> 학습이 안정적으로 시작할 가능성이 높아진다.

이번장은 PyTorch에서 초기화 방법을 어떻게 적용하는가에 초점이 있다. D2L은 `nn.init.xavier_uniform_()`를 층별 초기화 예제로 사용한다.

## 9. Custom Initialization

우리가 원하는 규칙을 직접 만들 수도 있다. D2L에선 이런 특이한 초기화를 예제로 쓴다.

```text
-10 ~ -5 -> 랜덤 값 유지
 -5 ~  5 -> 0
  5 ~ 10 -> 랜덤 값 유지
```

절댓값이 5보다 작은 weight는 모두 0으로 만든다.

D2L에서 원래 분포는 확률 1/4로 `U(5,10)`, 확률 1/2로 0, 확률 1/4로 `U(-10,-5)`가 되도록 정의되어 있다.

In [13]:
def my_init(module):
    if isinstance(module, nn.Linear):
        
        nn.init.uniform_(
            module.weight,
            -10,
            10
        )
        
        module.weight.data *= (
            module.weight.data.abs() >= 5
        )

In [14]:
net.apply(my_init)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [15]:
print(net[0].weight[:2])

tensor([[-9.9416,  0.0000, -0.0000,  0.0000],
        [-8.2066,  7.4243, -7.3405, -0.0000]], grad_fn=<SliceBackward0>)


## 10. Parameter 직접 수정하기

초기화 함수를 쓰지않고 weight 값을 직접 수정할 수도 있다.

예를 들어서 특정 weight 값을 42로 바꾸는 것도 가능하다. 하지만 일반 학습 과정에서 parameter를 직접 수정하는 것은 주의해야 한다.

초기화나 특별한 실험처럼 명확한 목적이 있을 때 사용한다.

In [ ]:
with torch.no_grad(): # 학습용 계산 그래프 기록 X
    net[0].weight += 1
    net[0].weight[0, 0] = 42

In [18]:
print(net[0].weight[0])

tensor([42.,  1.,  1.,  1.], grad_fn=<SelectBackward0>)


## 11. 전체 흐름

초기화 함수 이름을 전부 외우는 것이 목적이 아니다.

결국 핵심 구조는 이렇다.

```text
모델 생성
   ↓
Parameter 생성
   ↓
Weight / Bias 초기값 설정
   ↓
학습 시작
   ↓
Forward
   ↓
Loss
   ↓
Backward
   ↓
Gradient
   ↓
Optimizer
   ↓
Parameter 업데이트
```

초기화는 모델을 만든 직후, 학습을 시작하기 전에 한 번 수행한다.

## 12. 오늘의 정리

- Parameter Initialization은 학습 시작 전 weight와 bias의 초기값을 정하는 것이다.
- PyTorch는 `nn.Linear`를 만들 때 기본적으로 parameter를 초기화해준다.
- `torch.nn.init`을 사용하면 원하는 방식으로 직접 초기화할 수 있다.
- `nn.init.normal_()`은 정규분포를 이용해 초기화한다.
- `nn.init.zeros_()`은 값을 모두 0으로 만든다.
- `nn.init.constant_()`은 원하는 상수로 초기화한다.
- `nn.init.xavier_uniform_()`은 층의 크기를 고려한 Xavier 초기화를 수행한다.
- `net.apply(function)`을 사용하면 모델 내부의 여러 층에 초기화 함수를 적용할 수 있다.
- 필요하면 층마다 서로 다른 초기화 방법을 사용할 수도 있다.
- 직접 사용자 정의 초기화 규칙을 만드는 것도 가능하다.
- 초기화는 학습 전에 출발점을 정하는 것이고, optimizer는 학습 중 gradient를 이용해 parameter를 수정한다.